# 05 — Visualization Dashboard

Comprehensive visualisations for the forecasting pipeline:

1. **Price & regime timeline** — coloured by regime state
2. **Survival curve** — P(price > X) at many price levels
3. **Trajectory forecast** — day-by-day expected price with CI ribbons
4. **Backtest comparison** — walk-forward results across tickers and horizons
5. **Quality heatmap** — score matrix by ticker × horizon
6. **Signal correlation** — PM signal vs realised returns

**Prerequisites:**
```bash
conda activate cramer-research
cd ../research && pip install -e . && cd ../notebooks
```

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from research.data.prices import fetch_historical_prices
from research.models.markov import (
    classify_regime_series,
    estimate_transition_matrix,
    compute_markov_forecast,
)
from research.viz.distributions import plot_regime_sequence, plot_trajectory

## 1. Price & Regime Timeline

Multi-ticker comparison with regime colouring.

In [ ]:
TICKERS = ['BTC', 'ETH', 'SOL']
DAYS = 180

fig, axes = plt.subplots(len(TICKERS), 1, figsize=(14, 3 * len(TICKERS)), sharex=True)
if len(TICKERS) == 1:
    axes = [axes]

for ax, ticker in zip(axes, TICKERS):
    prices = fetch_historical_prices(ticker, days=DAYS)
    prices['returns'] = prices['close'].pct_change()
    returns = prices['returns'].dropna().values
    regimes = classify_regime_series(returns)
    
    colors = {'bull': 'green', 'bear': 'red', 'sideways': 'gray'}
    dates = prices['date'].values
    closes = prices['close'].values
    
    for i in range(len(regimes)):
        color = colors.get(regimes[i], 'blue')
        if i < len(dates) - 1:
            ax.plot(dates[i:i+2], closes[i:i+2], color=color, linewidth=1)
    
    ax.set_ylabel(ticker)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date')
fig.suptitle('Price & Regime Timeline', y=1.01)
plt.tight_layout()
plt.show()

## 2. Survival Curve

P(price > X) at many price levels — the full forecast distribution.

In [ ]:
TICKER = 'BTC'
HORIZON = 7

prices = fetch_historical_prices(TICKER, days=120)
current_price = float(prices['close'].iloc[-1])
returns = prices['close'].pct_change().dropna().values
regimes = classify_regime_series(returns)
P_mat = estimate_transition_matrix(regimes, decay_rate=0.97)
forecast = compute_markov_forecast(P_mat, regimes[-1], HORIZON)

# Compute regime up-rates
def _up_rates(regimes, returns, horizon, decay=0.97):
    counts = {s: {'up': 0.0, 'total': 0.0} for s in ['bull', 'bear', 'sideways']}
    max_start = min(len(regimes), len(returns)) - horizon
    for i in range(max_start):
        r = regimes[i]
        cr = sum(returns[i+1:i+1+horizon])
        w = decay ** (max_start - 1 - i)
        counts[r]['total'] += w
        if cr > 0: counts[r]['up'] += w
    return {s: counts[s]['up']/counts[s]['total'] if counts[s]['total'] > 0 else 0.5
            for s in ['bull', 'bear', 'sideways']}

up = _up_rates(regimes, returns, HORIZON)
p_up = sum(forecast[s] * up[s] for s in ['bull', 'bear', 'sideways'])

# Build survival curve via Gaussian approximation
sigma_est = returns.std() * np.sqrt(HORIZON)
expected_return = p_up - 0.5
expected_price = current_price * (1 + expected_return)

price_levels = np.linspace(current_price * 0.7, current_price * 1.3, 100)
from scipy import stats
z = (price_levels - expected_price) / (sigma_est * current_price)
survival_probs = 1 - stats.norm.cdf(z)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(price_levels, survival_probs, linewidth=2.5, label='P(price > X)')
ax.axvline(current_price, color='red', linestyle='--', label=f'Current: ${current_price:,.0f}')
ax.axvline(expected_price, color='blue', linestyle='--', label=f'Expected: ${expected_price:,.0f}')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Price')
ax.set_ylabel('Survival Probability')
ax.set_title(f'{TICKER} {HORIZON}-Day Survival Curve')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 3. Trajectory Forecast with CI Ribbons

Day-by-day expected price and confidence intervals.

In [ ]:
MAX_HORIZON = 14
trajectory_rows = []

for day in range(1, MAX_HORIZON + 1):
    f = compute_markov_forecast(P_mat, regimes[-1], day)
    u = _up_rates(regimes, returns, day)
    p_up_day = sum(f[s] * u[s] for s in ['bull', 'bear', 'sideways'])
    expected_ret = p_up_day - 0.5
    expected = current_price * (1 + expected_ret)
    sigma_day = returns.std() * np.sqrt(day)
    lower = current_price * (1 + expected_ret - 1.645 * sigma_day)
    upper = current_price * (1 + expected_ret + 1.645 * sigma_day)
    trajectory_rows.append({
        'day': day,
        'expected': expected,
        'lower': lower,
        'upper': upper,
        'p_up': p_up_day,
    })

traj_df = pd.DataFrame(trajectory_rows)

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(traj_df['day'], traj_df['lower'], traj_df['upper'], alpha=0.2, color='steelblue', label='90% CI')
ax.plot(traj_df['day'], traj_df['expected'], linewidth=2.5, color='blue', label='Expected')
ax.axhline(current_price, color='red', linestyle='--', label=f'Current: ${current_price:,.0f}')
ax.set_xlabel('Day')
ax.set_ylabel('Price')
ax.set_title(f'{TICKER}: {MAX_HORIZON}-Day Trajectory Forecast')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(traj_df[['day', 'expected', 'lower', 'upper', 'p_up']].to_string(index=False))

## 4. Backtest Comparison

Simulate walk-forward backtest results across tickers and horizons.
Uses the same regime model to generate directional signals.

In [ ]:
def walk_forward_backtest(prices, horizon, decay=0.97, warmup=60, stride=7):
    """Simple walk-forward backtest using regime signals."""
    closes = prices['close'].values
    returns = prices['close'].pct_change().dropna().values
    dates = prices['date'].iloc[1:].values
    
    predictions = []
    actuals = []
    
    for i in range(warmup, len(returns) - horizon, stride):
        window_returns = returns[i-warmup:i]
        window_regimes = classify_regime_series(window_returns)
        P = estimate_transition_matrix(window_regimes, decay_rate=decay)
        f = compute_markov_forecast(P, window_regimes[-1], horizon)
        u = _up_rates(window_regimes, window_returns, horizon, decay)
        p_up = sum(f[s] * u[s] for s in ['bull', 'bear', 'sideways'])
        
        pred_direction = 1 if p_up > 0.5 else -1
        actual_return = sum(returns[i:i+horizon])
        actual_direction = 1 if actual_return > 0 else -1
        
        predictions.append(pred_direction)
        actuals.append(actual_direction)
    
    correct = sum(1 for p, a in zip(predictions, actuals) if p == a)
    accuracy = correct / len(predictions) if predictions else 0
    return accuracy, len(predictions)

# Run backtests
backtest_results = []
for ticker in ['BTC', 'ETH', 'SOL']:
    for horizon in [7, 14, 30]:
        try:
            p = fetch_historical_prices(ticker, days=365)
            acc, n = walk_forward_backtest(p, horizon)
            backtest_results.append({
                'ticker': ticker,
                'horizon': horizon,
                'accuracy': acc,
                'n': n,
            })
        except Exception:
            backtest_results.append({'ticker': ticker, 'horizon': horizon, 'accuracy': np.nan, 'n': 0})

bt_df = pd.DataFrame(backtest_results)
bt_pivot = bt_df.pivot(index='ticker', columns='horizon', values='accuracy')

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(bt_pivot, annot=True, fmt='.1%', cmap='RdYlGn', vmin=0.3, vmax=0.7,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Directional Accuracy'})
ax.set_title('Walk-Forward Backtest: Directional Accuracy')
ax.set_xlabel('Horizon (days)')
ax.set_ylabel('Ticker')
plt.tight_layout()
plt.show()

## 5. Quality Score Heatmap

Quality score matrix by ticker × horizon.

In [ ]:
from research.models.ensemble import compute_quality_score, score_to_grade

# Simulate quality scores
quality_results = []
for ticker in ['BTC', 'ETH', 'SOL']:
    for horizon in [7, 14, 30]:
        # Fake metrics for demo
        n_markets = np.random.randint(3, 8)
        avg_qual = np.random.uniform(0.5, 0.9)
        sigma = 0.05 + horizon * 0.002
        signals = np.random.randint(2, 5)
        whales = np.random.randint(0, 2)
        score = compute_quality_score([], avg_qual, sigma, signals, whales)
        quality_results.append({
            'ticker': ticker,
            'horizon': horizon,
            'score': score,
            'grade': score_to_grade(score),
        })

q_df = pd.DataFrame(quality_results)
q_pivot = q_df.pivot(index='ticker', columns='horizon', values='score')

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(q_pivot, annot=True, fmt='.0f', cmap='RdYlGn_r', vmin=0, vmax=100,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Quality Score'})
ax.set_title('Quality Score by Ticker × Horizon')
ax.set_xlabel('Horizon (days)')
ax.set_ylabel('Ticker')
plt.tight_layout()
plt.show()

## 6. Signal Correlation

Scatter plot: Markov P(up) vs realised forward returns.

In [ ]:
TICKER = 'BTC'
HORIZON = 7

prices = fetch_historical_prices(TICKER, days=365)
returns = prices['close'].pct_change().dropna().values
regimes = classify_regime_series(returns)

pred_probs = []
realised_rets = []

for i in range(60, len(returns) - HORIZON, 7):
    window = returns[i-60:i]
    window_regimes = classify_regime_series(window)
    P = estimate_transition_matrix(window_regimes, decay_rate=0.97)
    f = compute_markov_forecast(P, window_regimes[-1], HORIZON)
    u = _up_rates(window_regimes, window, HORIZON)
    p_up = sum(f[s] * u[s] for s in ['bull', 'bear', 'sideways'])
    actual = sum(returns[i:i+HORIZON])
    
    pred_probs.append(p_up)
    realised_rets.append(actual)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(pred_probs, np.array(realised_rets) * 100, alpha=0.5, s=50)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

# Add regression line
z = np.polyfit(pred_probs, np.array(realised_rets) * 100, 1)
p = np.poly1d(z)
x_line = np.linspace(min(pred_probs), max(pred_probs), 100)
ax.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Regression (slope={z[0]:.2f})')

ax.set_xlabel('Predicted P(up)')
ax.set_ylabel(f'Realised {HORIZON}-Day Return (%)')
ax.set_title(f'{TICKER}: Predicted vs Realised Returns (H={HORIZON})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Directional accuracy
pred_dir = [1 if p > 0.5 else -1 for p in pred_probs]
actual_dir = [1 if r > 0 else -1 for r in realised_rets]
dir_acc = sum(1 for p, a in zip(pred_dir, actual_dir) if p == a) / len(pred_dir)
print(f"Directional accuracy: {dir_acc:.1%} (n={len(pred_dir)})")